# ۴. تخمین فشار در ویزیت بازگشت

این نوت‌بوک به تخمین فشار می‌رسد، ولی خط لولهٔ سیگنال خودمان را وارد مدل نمی‌کند. آشکارساز قله روی راه‌رفتن قابل اتکا نبود و ویژگی HRV نوری را هم از روی موج نساختیم.

ورودی، ستون‌های دلتای `features.tsv` منتشرشده است. این بازتولید بخش استخراج ویژگی مقاله نیست. مقایسهٔ مدل، تقسیم شرکت‌کننده، و دو خط پایه را می‌سنجد.

عدد اصلی هر خانه، میانگین بایاس و میانگین انحراف معیار روی ۱۰۰ نمونهٔ ۹۰درصدی از شرکت‌کننده‌هاست. خطا برابر پیش‌بینی منهای مرجع است. آلفا برابر ۱ روی ویژگی استانداردشده در هر فولد است؛ مقاله آلفا را نداده و این عدد تنظیم نویسنده نیست.

جدول مقاله برای تونومتری سمعی در کل جمعیت، SBP را حدود `0.32±9.75` و DBP را حدود `0.54±7.65` گزارش کرده است. این دو عدد از روی تصویر جدول خوانده شده‌اند. انتظار برابری با آن عددها را نداشته باشید: خط پایهٔ فایل «میانگین کالیبراسیون» است، آلفا مجهول است، و گروه نوری و ساعت هوشمند ستون HRV نوری را ندارند.


In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('خارج از کولب؛ اتصال درایو رد شد.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import base64
from pathlib import Path
root = Path('/content/aurorabp_reproduction')
(root / 'src').mkdir(parents=True, exist_ok=True)
(root / 'src' / '__init__.py').write_text('__version__ = "0.1.0"\n', encoding='utf-8')
(root / 'src' / 'bp_protocol.py').write_bytes(base64.b64decode("IiIiUmV0dXJuLXZpc2l0IGJsb29kIHByZXNzdXJlIHByb3RvY29sIGZvciB0aGUgQXVyb3JhLUJQIHJlcHJvZHVjdGlvbi4KClRoZSBzZW5zb3IgZmVhdHVyZXMgYXJlIHRoZSByZWxlYXNlZCBkZWx0YSBjb2x1bW5zLCBub3QgZmVhdHVyZXMgY29tcHV0ZWQKYnkgdGhpcyBwcm9qZWN0LiBSaWRnZSBhbHBoYSwgZXJyb3IgZGVncmVlcyBvZiBmcmVlZG9tLCBhbmQgdGhlIEVDRyBxdWFsaXR5CnJ1bGUgYXJlIG5vdCBhdXRob3Igc2V0dGluZ3MuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBJdGVyYWJsZSwgTGlzdCwgT3B0aW9uYWwKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgojIFtERUNJU0lPTl0gU3RhbmRhcmRpemVkIGlucHV0cywgdW5wZW5hbGl6ZWQgaW50ZXJjZXB0LCBhbHBoYSAxLgojIFRoZSBwYXBlciBkb2VzIG5vdCBnaXZlIGFscGhhLiBUaGUgZXhhbXBsZSBub3RlYm9vayB1c2VzIDAuMDAxIG9uIHJhdyB1bml0cy4KUklER0VfQUxQSEEgPSAxLjAKUVVBTElUWV9USFJFU0hPTEQgPSAwLjY1ICAjIFtHSVRIVUJdIGRvY3M6IHVzZWQgaW4gdGhlIHN0dWR5J3MgcHJlZGljdGlvbiBleHBlcmltZW50cwpCT09UU1RSQVBfUkVQTElDQVRFUyA9IDEwMCAgIyBbUEFQRVJdCkJPT1RTVFJBUF9GUkFDVElPTiA9IDAuOTAgICMgW1BBUEVSXQojIFtERUNJU0lPTl0gVGhlIHBhcGVyIGRvZXMgbm90IHNheSBzYW1wbGluZyB3aXRoIHJlcGxhY2VtZW50LgpCT09UU1RSQVBfV0lUSF9SRVBMQUNFTUVOVCA9IEZhbHNlCllPVU5HX0FHRV9NQVggPSAzNSAgIyBbUEFQRVJdCiMgW1BBUEVSXSBtZXRob2RzLiBGaWd1cmUgMyBjYXB0aW9uIHVzZXMgYSBzdHJpY3QgaW5lcXVhbGl0eS4KTk9STU9URU5TSVZFX1JFU1RJTkdfU0JQX01BWCA9IDEzMC4wCkVYRVJDSVNFX1BBVFRFUk4gPSByZS5jb21waWxlKHIiZXhlcmNpc2V8cnVubmluZ3x3YWxraW5nfGNvb2wuP2Rvd24iLCByZS5JR05PUkVDQVNFKQoKIyBUYWJsZSBJViwgdXNpbmcgdGhlIHJlbGVhc2VkIGRlbHRhIGNvbHVtbnMuIFtQQVBFUl0gZ3JvdXAgbWVtYmVyc2hpcC4KIyBbR0lUSFVCXSB0aGUgZXhhbXBsZSBtb2RlbCBwcmVkaWN0cyBkZWx0YV9zYnAgZnJvbSBkZWx0YSBmZWF0dXJlcy4KIyBUaW1lLW9mLWRheSB0ZXJtcyBhcmUgYWJzb2x1dGUsIG5vdCBkZWx0YXMuIFtQQVBFUl0KRkVBVFVSRV9HUk9VUFM6IERpY3Rbc3RyLCBMaXN0W3N0cl1dID0gewogICAgImVjZyI6IFsKICAgICAgICAiZGVsdGFfYmVhdF9sZW5ndGhfZWtnIiwKICAgICAgICAiZGVsdGFfaHJfZWtnIiwKICAgICAgICAiZGVsdGFfaHJ2X2VrZyIsCiAgICAgICAgImNvc19ob3VyIiwKICAgICAgICAic2luX2hvdXIiLAogICAgXSwKICAgICJvcHRpY2FsIjogWwogICAgICAgICJkZWx0YV9hcm1fYW5nbGUiLAogICAgICAgICJkZWx0YV9iZWF0X2xlbmd0aF9la2ciLAogICAgICAgICJkZWx0YV9kcGR0X29wdGljYWwiLAogICAgICAgICJkZWx0YV9ocl9la2ciLAogICAgICAgICJkZWx0YV9ocl9vcHRpY2FsIiwKICAgICAgICAiZGVsdGFfaHJ2X2VrZyIsCiAgICAgICAgImRlbHRhX3F1YWxpdHlfb3B0aWNhbCIsCiAgICAgICAgImRlbHRhX2ludnBhdF9vcHRpY2FsIiwKICAgICAgICAiZGVsdGFfcnBhdF9vcHRpY2FsIiwKICAgICAgICAiY29zX2hvdXIiLAogICAgICAgICJzaW5faG91ciIsCiAgICBdLAogICAgInNtYXJ0d2F0Y2giOiBbCiAgICAgICAgImRlbHRhX2FybV9hbmdsZSIsCiAgICAgICAgImRlbHRhX2RwZHRfb3B0aWNhbCIsCiAgICAgICAgImRlbHRhX2hyX29wdGljYWwiLAogICAgICAgICJkZWx0YV9xdWFsaXR5X29wdGljYWwiLAogICAgICAgICJjb3NfaG91ciIsCiAgICAgICAgInNpbl9ob3VyIiwKICAgIF0sCiAgICAidG9ub21ldHJpYyI6IFsKICAgICAgICAiZGVsdGFfYXJtX2FuZ2xlIiwKICAgICAgICAiZGVsdGFfYWl4X3ByZXNzdXJlIiwKICAgICAgICAiZGVsdGFfYmVhdF9sZW5ndGhfZWtnIiwKICAgICAgICAiZGVsdGFfZGljcm90aWNfbm90Y2hfaGVpZ2h0X3ByZXNzdXJlIiwKICAgICAgICAiZGVsdGFfZHBkdF9wcmVzc3VyZSIsCiAgICAgICAgImRlbHRhX2VqZWN0aW9uX2R1cmF0aW9uX3ByZXNzdXJlIiwKICAgICAgICAiZGVsdGFfZWplY3Rpb25fZHVyYXRpb25fZnJhY3Rpb25fcHJlc3N1cmUiLAogICAgICAgICJkZWx0YV9ocl9la2ciLAogICAgICAgICJkZWx0YV9ocl9wcmVzc3VyZSIsCiAgICAgICAgImRlbHRhX2hydl9la2ciLAogICAgICAgICJkZWx0YV9xdWFsaXR5X3ByZXNzdXJlIiwKICAgICAgICAiZGVsdGFfaW52cGF0X3ByZXNzdXJlIiwKICAgICAgICAiZGVsdGFfcnBhdF9wcmVzc3VyZSIsCiAgICAgICAgImRlbHRhX3J3YXRfcHJlc3N1cmUiLAogICAgICAgICJkZWx0YV9zeXNyaXNlX3ByZXNzdXJlIiwKICAgICAgICAiY29zX2hvdXIiLAogICAgICAgICJzaW5faG91ciIsCiAgICBdLAp9CgojIFtHSVRIVUJdIFRoZSByZWxlYXNlZCB0YWJsZSBoYXMgbm8gb3B0aWNhbCBvciB0b25vbWV0cmljIEhSViBjb2x1bW4uCiMgVGFibGUgSVYgYXNrcyBmb3Igb3B0aWNhbCBIUlYgaW4gdGhlIG9wdGljYWwgYW5kIHNtYXJ0d2F0Y2ggZ3JvdXBzLgpNSVNTSU5HX1RBQkxFX0ZFQVRVUkVTID0gewogICAgIm9wdGljYWwiOiBbImhydl9vcHRpY2FsIl0sCiAgICAic21hcnR3YXRjaCI6IFsiaHJ2X29wdGljYWwiXSwKfQoKIyBbREVDSVNJT05dIDAuNjUgaXMgZG9jdW1lbnRlZCBvbiB0aGUgb3B0aWNhbCBhbmQgdG9ub21ldHJpYyBxdWFsaXR5IGNvbHVtbnMuCiMgVGhlIEVDRyBncm91cCBoYXMgbm8gcXVhbGl0eSB0ZXJtLCBhbmQgRUNHIGF2YWlsYWJpbGl0eSBpbiB0aGUgcGFwZXIgaXMgaGlnaGVyCiMgdGhhbiB0aGUgb3RoZXIgc2Vuc29ycywgc28gdGhhdCB0aHJlc2hvbGQgaXMgbm90IGFwcGxpZWQgdG8gdGhlIEVDRyBncm91cC4KUVVBTElUWV9DT0xVTU4gPSB7CiAgICAiZWNnIjogTm9uZSwKICAgICJvcHRpY2FsIjogInF1YWxpdHlfb3B0aWNhbCIsCiAgICAic21hcnR3YXRjaCI6ICJxdWFsaXR5X29wdGljYWwiLAogICAgInRvbm9tZXRyaWMiOiAicXVhbGl0eV9wcmVzc3VyZSIsCn0KClNUQVRJQ19GRUFUVVJFUyA9IFsiYmFzZWxpbmVfc2JwIiwgImJhc2VsaW5lX2RicCIsICJjb3NfaG91ciIsICJzaW5faG91ciJdCgoKZGVmIGFkZF90aW1lX2ZlYXR1cmVzKGZyYW1lOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkNsb2NrIGhvdXIgZnJvbSB0aGUgdGltZXN0YW1wLCB3aXRob3V0IHRpbWV6b25lIGNvbnZlcnNpb24uIFtERUNJU0lPTl0iIiIKICAgIG91dHB1dCA9IGZyYW1lLmNvcHkoKQogICAgc3RhbXBzID0gcGQudG9fZGF0ZXRpbWUob3V0cHV0WyJkYXRlX3RpbWUiXSwgZXJyb3JzPSJjb2VyY2UiKQogICAgaG91ciA9IHN0YW1wcy5kdC5ob3VyICsgc3RhbXBzLmR0Lm1pbnV0ZSAvIDYwLjAgKyBzdGFtcHMuZHQuc2Vjb25kIC8gMzYwMC4wCiAgICBhbmdsZSA9IDIuMCAqIG5wLnBpICogaG91ciAvIDI0LjAKICAgIG91dHB1dFsiY29zX2hvdXIiXSA9IG5wLmNvcyhhbmdsZSkKICAgIG91dHB1dFsic2luX2hvdXIiXSA9IG5wLnNpbihhbmdsZSkKICAgIHJldHVybiBvdXRwdXQKCgpkZWYgZXhlcmNpc2VfbWFzayhtZWFzdXJlbWVudHM6IHBkLlNlcmllcykgLT4gcGQuU2VyaWVzOgogICAgcmV0dXJuIG1lYXN1cmVtZW50cy5hc3R5cGUoc3RyKS5zdHIuY29udGFpbnMoRVhFUkNJU0VfUEFUVEVSTiwgbmE9RmFsc2UpCgoKZGVmIGxvcG9fcHJlZGljdCh2YWx1ZXM6IG5wLm5kYXJyYXksIHRhcmdldDogbnAubmRhcnJheSwgZ3JvdXBzOiBucC5uZGFycmF5LCBhbHBoYTogZmxvYXQgPSBSSURHRV9BTFBIQSkgLT4gbnAubmRhcnJheToKICAgICIiIkxlYXZlLW9uZS1wYXJ0aWNpcGFudC1vdXQgcmlkZ2UuIFRyYWluLWZvbGQgc3RhbmRhcmRpemF0aW9uLiBbREVDSVNJT05dIGFscGhhLiIiIgogICAgZmVhdHVyZXMgPSBucC5hc2FycmF5KHZhbHVlcywgZHR5cGU9ZmxvYXQpCiAgICBvYnNlcnZlZCA9IG5wLmFzYXJyYXkodGFyZ2V0LCBkdHlwZT1mbG9hdCkKICAgIGxhYmVscyA9IG5wLmFzYXJyYXkoZ3JvdXBzKQogICAgcHJlZGljdGVkID0gbnAuZnVsbChvYnNlcnZlZC5zaGFwZSwgbnAubmFuKQogICAgcGVuYWx0eSA9IGFscGhhICogbnAuZXllKGZlYXR1cmVzLnNoYXBlWzFdKQogICAgZm9yIGxhYmVsIGluIHBkLnVuaXF1ZShsYWJlbHMpOgogICAgICAgIHRlc3QgPSBsYWJlbHMgPT0gbGFiZWwKICAgICAgICB0cmFpbiA9IH50ZXN0CiAgICAgICAgaWYgaW50KHRyYWluLnN1bSgpKSA8IGZlYXR1cmVzLnNoYXBlWzFdICsgMToKICAgICAgICAgICAgY29udGludWUKICAgICAgICB0cmFpbl94ID0gZmVhdHVyZXNbdHJhaW5dCiAgICAgICAgY2VudGVyID0gdHJhaW5feC5tZWFuKGF4aXM9MCkKICAgICAgICBzY2FsZSA9IHRyYWluX3guc3RkKGF4aXM9MCkKICAgICAgICBzY2FsZVtzY2FsZSA8IDFlLThdID0gMS4wCiAgICAgICAgdHJhaW5feiA9ICh0cmFpbl94IC0gY2VudGVyKSAvIHNjYWxlCiAgICAgICAgdGFyZ2V0X2NlbnRlciA9IGZsb2F0KG9ic2VydmVkW3RyYWluXS5tZWFuKCkpCiAgICAgICAgY29lZmZpY2llbnQgPSBucC5saW5hbGcuc29sdmUoCiAgICAgICAgICAgIHRyYWluX3ouVCBAIHRyYWluX3ogKyBwZW5hbHR5LAogICAgICAgICAgICB0cmFpbl96LlQgQCAob2JzZXJ2ZWRbdHJhaW5dIC0gdGFyZ2V0X2NlbnRlciksCiAgICAgICAgKQogICAgICAgIHByZWRpY3RlZFt0ZXN0XSA9ICgoZmVhdHVyZXNbdGVzdF0gLSBjZW50ZXIpIC8gc2NhbGUpIEAgY29lZmZpY2llbnQgKyB0YXJnZXRfY2VudGVyCiAgICByZXR1cm4gcHJlZGljdGVkCgoKZGVmIF9lcnJvcl9zdW1tYXJ5KHByZWRpY3RlZDogbnAubmRhcnJheSwgb2JzZXJ2ZWQ6IG5wLm5kYXJyYXkpIC0+IERpY3Rbc3RyLCBmbG9hdF06CiAgICBlcnJvciA9IHByZWRpY3RlZCAtIG9ic2VydmVkCiAgICBmaW5pdGUgPSBucC5pc2Zpbml0ZShlcnJvcikKICAgIGVycm9yID0gZXJyb3JbZmluaXRlXQogICAgaWYgZXJyb3Iuc2l6ZSA8IDI6CiAgICAgICAgcmV0dXJuIHsibiI6IGludChlcnJvci5zaXplKSwgImJpYXMiOiBucC5uYW4sICJzZCI6IG5wLm5hbn0KICAgIHJldHVybiB7CiAgICAgICAgIm4iOiBpbnQoZXJyb3Iuc2l6ZSksCiAgICAgICAgImJpYXMiOiBmbG9hdChlcnJvci5tZWFuKCkpLAogICAgICAgICJzZCI6IGZsb2F0KGVycm9yLnN0ZChkZG9mPTEpKSwKICAgIH0KCgpkZWYgX2Jvb3RzdHJhcF9pZHMoaWRlbnRpZmllcnM6IG5wLm5kYXJyYXksIHJlcGxpY2F0ZXM6IGludCwgZnJhY3Rpb246IGZsb2F0LCBzZWVkOiBpbnQpIC0+IEl0ZXJhYmxlW25wLm5kYXJyYXldOgogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBjb3VudCA9IGludChyb3VuZChpZGVudGlmaWVycy5zaXplICogZnJhY3Rpb24pKQogICAgY291bnQgPSBtaW4obWF4KGNvdW50LCAyKSwgaWRlbnRpZmllcnMuc2l6ZSkKICAgIGZvciBfIGluIHJhbmdlKHJlcGxpY2F0ZXMpOgogICAgICAgIHlpZWxkIHJuZy5jaG9pY2UoaWRlbnRpZmllcnMsIHNpemU9Y291bnQsIHJlcGxhY2U9Qk9PVFNUUkFQX1dJVEhfUkVQTEFDRU1FTlQpCgoKZGVmIGNvbXBsZXRlX2Nhc2VzKGZyYW1lOiBwZC5EYXRhRnJhbWUsIGNvbHVtbnM6IExpc3Rbc3RyXSwgdGFyZ2V0OiBzdHIpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlJvd3MgdXNlZCB0b2dldGhlciBmb3IgdGhlIGZlYXR1cmUgbW9kZWwgYW5kIHRoZSBwYWlyZWQgYmFzZWxpbmVzLiIiIgogICAgbmVlZGVkID0gbGlzdChkaWN0LmZyb21rZXlzKGNvbHVtbnMgKyBTVEFUSUNfRkVBVFVSRVMgKyBbdGFyZ2V0LCAicGlkIl0pKQogICAgcmV0dXJuIGZyYW1lLmRyb3BuYShzdWJzZXQ9bmVlZGVkKS5yZXNldF9pbmRleChkcm9wPVRydWUpCgoKZGVmIGV2YWx1YXRlX3Jvd3MoCiAgICBmcmFtZTogcGQuRGF0YUZyYW1lLAogICAgY29sdW1uczogTGlzdFtzdHJdLAogICAgdGFyZ2V0OiBzdHIsCiAgICByZXBsaWNhdGVzOiBpbnQgPSBCT09UU1RSQVBfUkVQTElDQVRFUywKICAgIHNlZWQ6IGludCA9IDIwMjIsCiAgICBmaXQ6IGJvb2wgPSBUcnVlLAopIC0+IERpY3Rbc3RyLCBmbG9hdF06CiAgICAiIiJNZWFzdXJlbWVudC1sZXZlbCBlcnJvci4gSGVhZGxpbmUgYmlhcyBhbmQgU0QgYXJlIG1lYW5zIG92ZXIgYm9vdHN0cmFwcy4gW1BBUEVSXSIiIgogICAgc3Vic2V0ID0gZnJhbWUKICAgIG9ic2VydmVkID0gc3Vic2V0W3RhcmdldF0udG9fbnVtcHkoZHR5cGU9ZmxvYXQpCiAgICBsYWJlbHMgPSBzdWJzZXRbInBpZCJdLnRvX251bXB5KCkKICAgIG1hdHJpeCA9IHN1YnNldFtjb2x1bW5zXS50b19udW1weShkdHlwZT1mbG9hdCkgaWYgZml0IGVsc2UgTm9uZQogICAgaWYgZml0OgogICAgICAgIHByZWRpY3RlZCA9IGxvcG9fcHJlZGljdChtYXRyaXgsIG9ic2VydmVkLCBsYWJlbHMpCiAgICBlbHNlOgogICAgICAgIHByZWRpY3RlZCA9IG5wLnplcm9zKG9ic2VydmVkLnNoYXBlLCBkdHlwZT1mbG9hdCkKICAgIGZ1bGwgPSBfZXJyb3Jfc3VtbWFyeShwcmVkaWN0ZWQsIG9ic2VydmVkKQogICAgYmlhc2VzOiBMaXN0W2Zsb2F0XSA9IFtdCiAgICBkZXZpYXRpb25zOiBMaXN0W2Zsb2F0XSA9IFtdCiAgICBpZGVudGlmaWVycyA9IG5wLnVuaXF1ZShsYWJlbHMpCiAgICBmb3IgY2hvc2VuIGluIF9ib290c3RyYXBfaWRzKGlkZW50aWZpZXJzLCByZXBsaWNhdGVzLCBCT09UU1RSQVBfRlJBQ1RJT04sIHNlZWQpOgogICAgICAgIGNob3Nlbl9yb3dzID0gbnAuaXNpbihsYWJlbHMsIGNob3NlbikKICAgICAgICBpZiBpbnQoY2hvc2VuX3Jvd3Muc3VtKCkpIDwgMjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBmaXQ6CiAgICAgICAgICAgIGZvbGRfcHJlZGljdGlvbiA9IGxvcG9fcHJlZGljdChtYXRyaXhbY2hvc2VuX3Jvd3NdLCBvYnNlcnZlZFtjaG9zZW5fcm93c10sIGxhYmVsc1tjaG9zZW5fcm93c10pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZm9sZF9wcmVkaWN0aW9uID0gbnAuemVyb3MoaW50KGNob3Nlbl9yb3dzLnN1bSgpKSwgZHR5cGU9ZmxvYXQpCiAgICAgICAgc3VtbWFyeSA9IF9lcnJvcl9zdW1tYXJ5KGZvbGRfcHJlZGljdGlvbiwgb2JzZXJ2ZWRbY2hvc2VuX3Jvd3NdKQogICAgICAgIGlmIG5wLmlzZmluaXRlKHN1bW1hcnlbImJpYXMiXSkgYW5kIG5wLmlzZmluaXRlKHN1bW1hcnlbInNkIl0pOgogICAgICAgICAgICBiaWFzZXMuYXBwZW5kKHN1bW1hcnlbImJpYXMiXSkKICAgICAgICAgICAgZGV2aWF0aW9ucy5hcHBlbmQoc3VtbWFyeVsic2QiXSkKICAgIGJpYXNfYXJyYXkgPSBucC5hc2FycmF5KGJpYXNlcywgZHR5cGU9ZmxvYXQpCiAgICBzZF9hcnJheSA9IG5wLmFzYXJyYXkoZGV2aWF0aW9ucywgZHR5cGU9ZmxvYXQpCiAgICByZXR1cm4gewogICAgICAgICJuX3Jvd3MiOiBpbnQobGVuKHN1YnNldCkpLAogICAgICAgICJuX3BhcnRpY2lwYW50cyI6IGludChpZGVudGlmaWVycy5zaXplKSwKICAgICAgICAibG9wb19iaWFzIjogZnVsbFsiYmlhcyJdLAogICAgICAgICJsb3BvX3NkIjogZnVsbFsic2QiXSwKICAgICAgICAiYm9vdF9iaWFzIjogZmxvYXQoYmlhc19hcnJheS5tZWFuKCkpIGlmIGJpYXNfYXJyYXkuc2l6ZSBlbHNlIG5wLm5hbiwKICAgICAgICAiYm9vdF9zZCI6IGZsb2F0KHNkX2FycmF5Lm1lYW4oKSkgaWYgc2RfYXJyYXkuc2l6ZSBlbHNlIG5wLm5hbiwKICAgICAgICAiYm9vdF9iaWFzX3AyXzUiOiBmbG9hdChucC5wZXJjZW50aWxlKGJpYXNfYXJyYXksIDIuNSkpIGlmIGJpYXNfYXJyYXkuc2l6ZSBlbHNlIG5wLm5hbiwKICAgICAgICAiYm9vdF9iaWFzX3A5N181IjogZmxvYXQobnAucGVyY2VudGlsZShiaWFzX2FycmF5LCA5Ny41KSkgaWYgYmlhc19hcnJheS5zaXplIGVsc2UgbnAubmFuLAogICAgICAgICJib290X3JlcGxpY2F0ZXNfdXNlZCI6IGludChiaWFzX2FycmF5LnNpemUpLAogICAgfQoKCmRlZiByZXN0aW5nX3N1cGluZV9zYnAoZmVhdHVyZXM6IHBkLkRhdGFGcmFtZSkgLT4gT3B0aW9uYWxbcGQuU2VyaWVzXToKICAgICIiIk1lZGlhbiBpbml0aWFsIHN1cGluZSBTQlAuIFtERUNJU0lPTl0gVGhlIHBhcGVyIHNheXMgcmVzdGluZyBTQlAgYW5kIGRvZXMgbm90IG5hbWUgdGhpcyBydWxlLiIiIgogICAgaW5pdGlhbCA9IGZlYXR1cmVzLmxvY1tmZWF0dXJlc1sicGhhc2UiXS5hc3R5cGUoc3RyKSA9PSAiaW5pdGlhbCJdCiAgICBzdXBpbmUgPSBpbml0aWFsLmxvY1tpbml0aWFsWyJtZWFzdXJlbWVudCJdLmFzdHlwZShzdHIpLnN0ci5jb250YWlucygic3VwaW5lIiwgY2FzZT1GYWxzZSwgbmE9RmFsc2UpXQogICAgaWYgc3VwaW5lLmVtcHR5IG9yICJzYnAiIG5vdCBpbiBzdXBpbmUuY29sdW1uczoKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIHN1cGluZS5ncm91cGJ5KCJwaWQiKVsic2JwIl0ubWVkaWFuKCkKCgpkZWYgcHJlcGFyZV9yZXR1cm5fdmlzaXQoZmVhdHVyZXM6IHBkLkRhdGFGcmFtZSkgLT4gdHVwbGVbcGQuRGF0YUZyYW1lLCBwZC5TZXJpZXNdOgogICAgZnJhbWUgPSBhZGRfdGltZV9mZWF0dXJlcyhmZWF0dXJlcykKICAgIHJldHVybl9yb3dzID0gZnJhbWUubG9jW2ZyYW1lWyJwaGFzZSJdLmFzdHlwZShzdHIpID09ICJyZXR1cm4iXS5jb3B5KCkKICAgIGRyb3BwZWQgPSBleGVyY2lzZV9tYXNrKHJldHVybl9yb3dzWyJtZWFzdXJlbWVudCJdKQogICAga2VwdCA9IHJldHVybl9yb3dzLmxvY1t+ZHJvcHBlZF0uY29weSgpCiAgICByZXR1cm4ga2VwdCwgcmV0dXJuX3Jvd3MubG9jW2Ryb3BwZWQsICJtZWFzdXJlbWVudCJdLmFzdHlwZShzdHIpCgoKZGVmIHBvcHVsYXRpb25fZnJhbWUoZnJhbWU6IHBkLkRhdGFGcmFtZSwgcGFydGljaXBhbnRzOiBwZC5EYXRhRnJhbWUsIHBvcHVsYXRpb246IHN0ciwgcmVzdGluZzogT3B0aW9uYWxbcGQuU2VyaWVzXSkgLT4gcGQuRGF0YUZyYW1lOgogICAgaWYgcG9wdWxhdGlvbiA9PSAiYWxsIjoKICAgICAgICByZXR1cm4gZnJhbWUKICAgIGlmIHBvcHVsYXRpb24gPT0gInlvdW5nIjoKICAgICAgICBhZ2VzID0gcGFydGljaXBhbnRzLmxvY1twYXJ0aWNpcGFudHNbImFnZSJdLm5vdG5hKCksIFsicGlkIiwgImFnZSJdXQogICAgICAgIHlvdW5nID0gYWdlcy5sb2NbYWdlc1siYWdlIl0gPD0gWU9VTkdfQUdFX01BWCwgInBpZCJdCiAgICAgICAgcmV0dXJuIGZyYW1lLmxvY1tmcmFtZVsicGlkIl0uaXNpbih5b3VuZyldCiAgICBpZiBwb3B1bGF0aW9uID09ICJub3Jtb3RlbnNpdmUiOgogICAgICAgIGlmIHJlc3RpbmcgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIGZyYW1lLmlsb2NbMDowXQogICAgICAgIGtlcHQgPSByZXN0aW5nLmxvY1tyZXN0aW5nIDw9IE5PUk1PVEVOU0lWRV9SRVNUSU5HX1NCUF9NQVhdLmluZGV4CiAgICAgICAgcmV0dXJuIGZyYW1lLmxvY1tmcmFtZVsicGlkIl0uaXNpbihrZXB0KV0KICAgIHJhaXNlIFZhbHVlRXJyb3IocG9wdWxhdGlvbikKCgpkZWYgcXVhbGl0eV9maWx0ZXIoZnJhbWU6IHBkLkRhdGFGcmFtZSwgZ3JvdXA6IHN0cikgLT4gcGQuRGF0YUZyYW1lOgogICAgY29sdW1uID0gUVVBTElUWV9DT0xVTU5bZ3JvdXBdCiAgICBpZiBjb2x1bW4gaXMgTm9uZToKICAgICAgICByZXR1cm4gZnJhbWUKICAgIHZhbHVlcyA9IGZyYW1lW2NvbHVtbl0KICAgIHJldHVybiBmcmFtZS5sb2NbdmFsdWVzID49IFFVQUxJVFlfVEhSRVNIT0xEXQoKCmRlZiBydW5fcmV0dXJuX3Zpc2l0KAogICAgZmVhdHVyZXM6IHBkLkRhdGFGcmFtZSwKICAgIHBhcnRpY2lwYW50czogcGQuRGF0YUZyYW1lLAogICAgcmVwbGljYXRlczogaW50ID0gQk9PVFNUUkFQX1JFUExJQ0FURVMsCiAgICBzZWVkOiBpbnQgPSAyMDIyLAogICAgb3V0cHV0X3BhdGg6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAopIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkZpdCBzZXBhcmF0ZSByZXR1cm4tdmlzaXQgbW9kZWxzIGJ5IHByb3RvY29sLCBwb3B1bGF0aW9uLCBhbmQgZmVhdHVyZSBncm91cC4iIiIKICAgIHZpc2l0LCBkcm9wcGVkX25hbWVzID0gcHJlcGFyZV9yZXR1cm5fdmlzaXQoZmVhdHVyZXMpCiAgICByZXN0aW5nID0gcmVzdGluZ19zdXBpbmVfc2JwKGZlYXR1cmVzKQogICAgcm93czogTGlzdFtkaWN0XSA9IFtdCiAgICBwcm90b2NvbHMgPSB7CiAgICAgICAgImF1c2N1bHRhdG9yeSI6IHZpc2l0LmxvY1t2aXNpdFsicGlkIl0uYXN0eXBlKHN0cikuc3RyLnN0YXJ0c3dpdGgoImEiKV0sCiAgICAgICAgIm9zY2lsbG9tZXRyaWMiOiB2aXNpdC5sb2NbdmlzaXRbInBpZCJdLmFzdHlwZShzdHIpLnN0ci5zdGFydHN3aXRoKCJvIildLAogICAgfQogICAgZm9yIHByb3RvY29sLCBwcm90b2NvbF9yb3dzIGluIHByb3RvY29scy5pdGVtcygpOgogICAgICAgIGZvciBwb3B1bGF0aW9uIGluICgiYWxsIiwgInlvdW5nIiwgIm5vcm1vdGVuc2l2ZSIpOgogICAgICAgICAgICBwZW9wbGUgPSBwb3B1bGF0aW9uX2ZyYW1lKHByb3RvY29sX3Jvd3MsIHBhcnRpY2lwYW50cywgcG9wdWxhdGlvbiwgcmVzdGluZykKICAgICAgICAgICAgaWYgcG9wdWxhdGlvbiA9PSAibm9ybW90ZW5zaXZlIiBhbmQgcmVzdGluZyBpcyBOb25lOgogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICAicHJvdG9jb2wiOiBwcm90b2NvbCwKICAgICAgICAgICAgICAgICAgICAgICAgInBvcHVsYXRpb24iOiBwb3B1bGF0aW9uLAogICAgICAgICAgICAgICAgICAgICAgICAicm93X3NldCI6ICJ1bmF2YWlsYWJsZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICJtb2RlbCI6ICJ1bmF2YWlsYWJsZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICJ0YXJnZXQiOiAiIiwKICAgICAgICAgICAgICAgICAgICAgICAgInJlbGlhYmlsaXR5IjogInJlc3Rpbmcgc3VwaW5lIFNCUCBjb3VsZCBub3QgYmUgaWRlbnRpZmllZCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJuX3Jvd3MiOiAwLAogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBncm91cCwgY29sdW1ucyBpbiBGRUFUVVJFX0dST1VQUy5pdGVtcygpOgogICAgICAgICAgICAgICAgYXZhaWxhYmxlID0gcXVhbGl0eV9maWx0ZXIocGVvcGxlLCBncm91cCkKICAgICAgICAgICAgICAgIG1pc3NpbmcgPSBNSVNTSU5HX1RBQkxFX0ZFQVRVUkVTLmdldChncm91cCwgW10pCiAgICAgICAgICAgICAgICBpZiBhdmFpbGFibGUuZW1wdHk6CiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZiJ7cHJvdG9jb2x9IHtwb3B1bGF0aW9ufSB7Z3JvdXB9OiBubyByb3dzIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgcmVsaWFiaWxpdHkgPSAib2ZmaWNpYWwgZmVhdHVyZXM7IGFscGhhIGFuZCBFQ0cgcXVhbGl0eSBydWxlIGFyZSBkZWNpc2lvbnMiCiAgICAgICAgICAgICAgICBpZiBtaXNzaW5nOgogICAgICAgICAgICAgICAgICAgIHJlbGlhYmlsaXR5ID0gIm1pc3NpbmcgIiArICIsICIuam9pbihtaXNzaW5nKSArICI7IG5vdCBhIGNvbXBsZXRlIFRhYmxlIElWIGdyb3VwIgogICAgICAgICAgICAgICAgaWYgcG9wdWxhdGlvbiA9PSAibm9ybW90ZW5zaXZlIjoKICAgICAgICAgICAgICAgICAgICByZWxpYWJpbGl0eSArPSAiOyByZXN0aW5nIFNCUCBydWxlIGlzIGEgZGVjaXNpb24iCiAgICAgICAgICAgICAgICBmb3IgdGFyZ2V0IGluICgiZGVsdGFfc2JwIiwgImRlbHRhX2RicCIpOgogICAgICAgICAgICAgICAgICAgIHVzYWJsZSA9IGNvbXBsZXRlX2Nhc2VzKGF2YWlsYWJsZSwgY29sdW1ucywgdGFyZ2V0KQogICAgICAgICAgICAgICAgICAgIGlmIHVzYWJsZVsicGlkIl0ubnVuaXF1ZSgpIDwgMTA6CiAgICAgICAgICAgICAgICAgICAgICAgIHByaW50KGYie3Byb3RvY29sfSB7cG9wdWxhdGlvbn0ge2dyb3VwfSB7dGFyZ2V0fTogZmV3ZXIgdGhhbiAxMCBwYXJ0aWNpcGFudHMiLCBmbHVzaD1UcnVlKQogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIHJpZGdlID0gZXZhbHVhdGVfcm93cyh1c2FibGUsIGNvbHVtbnMsIHRhcmdldCwgcmVwbGljYXRlcz1yZXBsaWNhdGVzLCBzZWVkPXNlZWQsIGZpdD1UcnVlKQogICAgICAgICAgICAgICAgICAgIHplcm8gPSBldmFsdWF0ZV9yb3dzKHVzYWJsZSwgY29sdW1ucywgdGFyZ2V0LCByZXBsaWNhdGVzPXJlcGxpY2F0ZXMsIHNlZWQ9c2VlZCwgZml0PUZhbHNlKQogICAgICAgICAgICAgICAgICAgIHN0YXRpYyA9IGV2YWx1YXRlX3Jvd3ModXNhYmxlLCBTVEFUSUNfRkVBVFVSRVMsIHRhcmdldCwgcmVwbGljYXRlcz1yZXBsaWNhdGVzLCBzZWVkPXNlZWQsIGZpdD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGZvciBtb2RlbCwgc3VtbWFyeSBpbiAoKCJyaWRnZSIsIHJpZGdlKSwgKCJ6ZXJvIiwgemVybyksICgic3RhdGljIiwgc3RhdGljKSk6CiAgICAgICAgICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJwcm90b2NvbCI6IHByb3RvY29sLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJwb3B1bGF0aW9uIjogcG9wdWxhdGlvbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAicm93X3NldCI6IGdyb3VwLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJtb2RlbCI6IG1vZGVsLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0YXJnZXQiOiB0YXJnZXQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1pc3NpbmdfdGFibGVfZmVhdHVyZXMiOiAiLCIuam9pbihtaXNzaW5nKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAicXVhbGl0eV9jb2x1bW4iOiBRVUFMSVRZX0NPTFVNTltncm91cF0gb3IgIiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImV4ZXJjaXNlX3Jvd3NfcmVtb3ZlZCI6IGludChkcm9wcGVkX25hbWVzLnNpemUpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJyZWxpYWJpbGl0eSI6IHJlbGlhYmlsaXR5LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICoqc3VtbWFyeSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgICAgICAgICBmIntwcm90b2NvbH0ge3BvcHVsYXRpb259IHtncm91cH0ge3RhcmdldH06ICIKICAgICAgICAgICAgICAgICAgICAgICAgZiJyaWRnZSB7cmlkZ2VbJ2Jvb3RfYmlhcyddOi4yZn3CsXtyaWRnZVsnYm9vdF9zZCddOi4yZn0gICIKICAgICAgICAgICAgICAgICAgICAgICAgZiJ6ZXJvIHt6ZXJvWydib290X2JpYXMnXTouMmZ9wrF7emVyb1snYm9vdF9zZCddOi4yZn0gICIKICAgICAgICAgICAgICAgICAgICAgICAgZiJzdGF0aWMge3N0YXRpY1snYm9vdF9iaWFzJ106LjJmfcKxe3N0YXRpY1snYm9vdF9zZCddOi4yZn0gICIKICAgICAgICAgICAgICAgICAgICAgICAgZiJuPXtyaWRnZVsnbl9yb3dzJ119IiwKICAgICAgICAgICAgICAgICAgICAgICAgZmx1c2g9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgaWYgb3V0cHV0X3BhdGg6CiAgICAgICAgICAgICAgICAgICAgICAgIHBkLkRhdGFGcmFtZShyb3dzKS50b19qc29uKG91dHB1dF9wYXRoLCBvcmllbnQ9InJlY29yZHMiLCBpbmRlbnQ9MikKICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykK"))
print(root / 'src' / 'bp_protocol.py')


/content/aurorabp_reproduction/src/bp_protocol.py


In [3]:
import sys
from pathlib import Path
sys.path.insert(0, '/content/aurorabp_reproduction')
import numpy as np
import pandas as pd
from src.bp_protocol import run_return_visit

candidates = [
    Path('/content/aurora_waveforms'),
    Path('/content/drive/MyDrive/aurora_dataset'),
]
data_dir = next(path for path in candidates if (path / 'features.tsv').is_file())
print('داده:', data_dir)
features = pd.read_csv(data_dir / 'features.tsv', sep='\t')
participants = pd.read_csv(data_dir / 'participants.tsv', sep='\t')
gap_sbp = (features['sbp'] - features['baseline_sbp'] - features['delta_sbp']).abs()
gap_dbp = (features['dbp'] - features['baseline_dbp'] - features['delta_dbp']).abs()
print(f'بیشینه اختلاف delta_sbp با sbp-baseline: {gap_sbp.max():.6f}')
print(f'بیشینه اختلاف delta_dbp با dbp-baseline: {gap_dbp.max():.6f}')
if gap_sbp.max() > 0.05 or gap_dbp.max() > 0.05:
    raise RuntimeError('تعریف دلتای فشار با ستون‌های فایل یکی نیست.')


داده: /content/aurora_waveforms
بیشینه اختلاف delta_sbp با sbp-baseline: 0.000000
بیشینه اختلاف delta_dbp با dbp-baseline: 0.000000


In [4]:
output = Path('/content/aurorabp_outputs/return_visit_bp.json')
output.parent.mkdir(parents=True, exist_ok=True)
results = run_return_visit(features, participants, output_path=str(output))
print(output)
results


auscultatory all ecg delta_sbp: ridge 0.01±10.53  zero 1.58±10.49  static -0.00±10.37  n=3154
auscultatory all ecg delta_dbp: ridge 0.00±8.15  zero 1.70±8.19  static -0.00±7.90  n=3153
auscultatory all optical delta_sbp: ridge 0.01±10.51  zero 1.67±10.53  static -0.00±10.42  n=2775
auscultatory all optical delta_dbp: ridge -0.00±7.81  zero 1.73±8.04  static -0.00±7.77  n=2774
auscultatory all smartwatch delta_sbp: ridge 0.02±10.53  zero 1.60±10.55  static -0.00±10.45  n=2793
auscultatory all smartwatch delta_dbp: ridge 0.00±7.90  zero 1.66±8.06  static -0.00±7.78  n=2792
auscultatory all tonometric delta_sbp: ridge 0.01±10.04  zero 1.49±10.40  static -0.00±10.27  n=2796
auscultatory all tonometric delta_dbp: ridge 0.00±8.02  zero 1.69±8.17  static 0.00±7.84  n=2795
auscultatory young ecg delta_sbp: ridge 0.01±8.98  zero 2.28±8.77  static -0.00±8.87  n=1047
auscultatory young ecg delta_dbp: ridge 0.02±8.58  zero 1.60±8.54  static 0.00±8.06  n=1046
auscultatory young optical delta_sbp: r

,protocol,population,row_set,model,target,missing_table_features,quality_column,exercise_rows_removed,reliability,n_rows,n_participants,lopo_bias,lopo_sd,boot_bias,boot_sd,boot_bias_p2_5,boot_bias_p97_5,boot_replicates_used
0,auscultatory,all,ecg,ridge,delta_sbp,,,0,official features; alpha and ECG quality rule ...,3154,635,0.005136,10.545430,0.005858,10.527033,0.003260,0.007881,100
1,auscultatory,all,ecg,zero,delta_sbp,,,0,official features; alpha and ECG quality rule ...,3154,635,1.563729,10.514299,1.577333,10.490117,1.325378,1.803286,100
2,auscultatory,all,ecg,static,delta_sbp,,,0,official features; alpha and ECG quality rule ...,3154,635,-0.002008,10.391086,-0.001925,10.369187,-0.004017,0.001271,100
3,auscultatory,all,ecg,ridge,delta_dbp,,,0,official features; alpha and ECG quality rule ...,3153,635,0.001853,8.151861,0.002093,8.154593,-0.000025,0.004948,100
4,auscultatory,all,ecg,zero,delta_dbp,,,0,official features; alpha and ECG quality rule ...,3153,635,1.690295,8.195246,1.700177,8.193978,1.489384,1.905367,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,oscillometric,normotensive,tonometric,zero,delta_sbp,,quality_pressure,0,official features; alpha and ECG quality rule ...,398,206,-7.806533,9.601060,-7.815573,9.599076,-8.146156,-7.397572,100
116,oscillometric,normotensive,tonometric,static,delta_sbp,,quality_pressure,0,official features; alpha and ECG quality rule ...,398,206,0.016077,9.345315,0.014898,9.355497,-0.036128,0.040985,100
117,oscillometric,normotensive,tonometric,ridge,delta_dbp,,quality_pressure,0,official features; alpha and ECG quality rule ...,398,206,-0.013747,7.662524,-0.016654,7.707999,-0.061812,0.032140,100
118,oscillometric,normotensive,tonometric,zero,delta_dbp,,quality_pressure,0,official features; alpha and ECG quality rule ...,398,206,-8.924623,7.498403,-8.940639,7.504003,-9.252957,-8.643309,100


In [5]:
show = results.loc[results['model'].isin(['ridge', 'zero', 'static']), [
    'protocol', 'population', 'row_set', 'model', 'target', 'n_rows', 'n_participants',
    'boot_bias', 'boot_sd', 'lopo_bias', 'lopo_sd', 'reliability',
]].copy()
for column in ('boot_bias', 'boot_sd', 'lopo_bias', 'lopo_sd'):
    show[column] = show[column].map(lambda value: round(float(value), 2) if pd.notna(value) else value)
headline = show.loc[(show['population'] == 'all') & (show['row_set'] == 'tonometric')]
print('تونومتری، کل جمعیت. boot_bias و boot_sd عدد گزارش‌اند. lopo فقط بررسی روی همهٔ شرکت‌کننده‌هاست و آمار مقاله نیست.')
headline


تونومتری، کل جمعیت. boot_bias و boot_sd عدد گزارش‌اند. lopo فقط بررسی روی همهٔ شرکت‌کننده‌هاست و آمار مقاله نیست.


,protocol,population,row_set,model,target,n_rows,n_participants,boot_bias,boot_sd,lopo_bias,lopo_sd,reliability
18,auscultatory,all,tonometric,ridge,delta_sbp,2796,632,0.01,10.04,0.01,10.00,official features; alpha and ECG quality rule ...
19,auscultatory,all,tonometric,zero,delta_sbp,2796,632,1.49,10.40,1.49,10.38,official features; alpha and ECG quality rule ...
20,auscultatory,all,tonometric,static,delta_sbp,2796,632,-0.00,10.27,-0.00,10.25,official features; alpha and ECG quality rule ...
21,auscultatory,all,tonometric,ridge,delta_dbp,2795,632,0.00,8.02,-0.00,8.00,official features; alpha and ECG quality rule ...
22,auscultatory,all,tonometric,zero,delta_dbp,2795,632,1.69,8.17,1.68,8.17,official features; alpha and ECG quality rule ...
23,auscultatory,all,tonometric,static,delta_dbp,2795,632,0.00,7.84,0.00,7.83,official features; alpha and ECG quality rule ...
66,oscillometric,all,tonometric,ridge,delta_sbp,577,310,-0.08,11.61,-0.07,11.57,official features; alpha and ECG quality rule ...
67,oscillometric,all,tonometric,zero,delta_sbp,577,310,-5.69,11.22,-5.70,11.23,official features; alpha and ECG quality rule ...
68,oscillometric,all,tonometric,static,delta_sbp,577,310,0.01,10.65,0.01,10.64,official features; alpha and ECG quality rule ...
69,oscillometric,all,tonometric,ridge,delta_dbp,577,310,-0.04,8.14,-0.03,8.14,official features; alpha and ECG quality rule ...
